In [1]:
import pandas as pd
from pulp import LpProblem, LpVariable, LpMaximize, lpSum, LpBinary, value

In [2]:
df = pd.read_csv("sample_portfolio.csv")
df

,applicant_id,loan_amount,expected_profit,default_probability
0,1,50000,8000,0.10
1,2,40000,6000,0.08
2,3,70000,9000,0.20
3,4,30000,5000,0.05
4,5,60000,8500,0.15
5,6,45000,6500,0.09


In [3]:
budget = 150000
max_avg_default_risk = 0.12

In [4]:
model = LpProblem("Loan_Approval_Optimization", LpMaximize)

x = {i: LpVariable(f"x_{i}", cat=LpBinary) for i in df.index}

In [5]:
model += lpSum(df.loc[i, "expected_profit"] * x[i] for i in df.index)

In [6]:
model += lpSum(df.loc[i, "loan_amount"] * x[i] for i in df.index) <= budget

In [7]:
model += lpSum(
    (df.loc[i, "default_probability"] - max_avg_default_risk) * x[i]
    for i in df.index
) <= 0

In [8]:
model.solve()
print("Solved")

Solved


In [9]:
selected = df[[value(x[i]) == 1 for i in df.index]].copy()
selected

,applicant_id,loan_amount,expected_profit,default_probability
0,1,50000,8000,0.10
1,2,40000,6000,0.08
4,5,60000,8500,0.15


In [10]:
print("Total Loan Amount:", selected["loan_amount"].sum())
print("Total Expected Profit:", selected["expected_profit"].sum())
print("Average Default Risk:", selected["default_probability"].mean())

Total Loan Amount: 150000
Total Expected Profit: 22500
Average Default Risk: 0.10999999999999999
